# Notebook 20 — Strict N07 FER replay for N17v2 neutral ablation

**UNEXECUTED / gated reproduction.** No diffusion, biometric or NIQE reruns. First test N07 recorded original predictions against a specified frozen Hugging Face ViT classifier. A failed original-prediction match blocks N17 scoring and requires N07 preprocessing inspection; never silently switch evaluators or remap labels to match. Source and prediction rows stay **PRIVATE** in Google Drive.

Both frozen model identifiers/revisions come from N07 `resolved_config.json`; `trpakov` is a *pre-registered provisional candidate* for the primary FER classifier, not yet independently proven to be the one generating `original_predicted_class`. The other ViT was used in N07 as additional evaluator and is not a fallback chosen to improve a metric. The exact N07 code/preprocessing must be reviewed for final provenance, even if the gate passes.


In [ ]:
%pip -q install 'transformers==4.44.2' 'huggingface-hub==0.36.2' 'pandas>=2.2,<3' 'Pillow>=10,<12' 'accelerate>=0.30,<2' 'torchvision'
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, numpy as np, pandas as pd, torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
RUNS=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=RUNS/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
N17=RUNS/'nb17v2_neutral_expression_12sources_2seeds_sd3pinned'
N18=RUNS/'nb18_paired_neutral_pilot_analysis_v1'
N19=RUNS/'nb19_n17v2_frozen_v3_biometrics'
OUT=RUNS/'nb20_exact_n07_fer_replay'; OUT.mkdir(exist_ok=True)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for x in iter(lambda: f.read(1024*1024),b''):h.update(x)
    return h.hexdigest()
raw=pd.read_csv(N07/'raw_results.csv')
receipts=pd.read_csv(N17/'private_execution_receipts.csv')
ref=pd.read_csv(N18/'private_paired_reference_and_n17_manifest.csv')
biom=pd.read_csv(N19/'private_v3_biometric_rows.csv')
assert len(receipts)==len(ref)==len(biom)==24
assert receipts.source_id.nunique()==12 and not receipts.duplicated(['source_id','seed']).any()
assert set(receipts.status)=={'VERIFIED'}
for r in receipts.itertuples(index=False):
    assert sha(r.output_path)==r.output_sha256
originals=ref[['source_id','original_fer_predicted_class']].drop_duplicates()
assert len(originals)==12 and originals.source_id.is_unique
print('PREFLIGHT PASS: 12 sources, 24 N17 images, N07 original recorded labels available')


In [ ]:
# N07 frozen metadata; explicit primary FER model candidate, NOT inferred from outputs.
resolved=json.loads((N07/'resolved_config.json').read_text())
frozen=resolved['evaluators']['emotion_model_revisions']
MODEL_ID='trpakov/vit-face-expression'
MODEL_REV='ef0bc6fc34241b6587e7e009e7711357be28c024'
assert frozen[MODEL_ID]==MODEL_REV
processor=AutoImageProcessor.from_pretrained(MODEL_ID,revision=MODEL_REV)
model=AutoModelForImageClassification.from_pretrained(MODEL_ID,revision=MODEL_REV)
model.eval()
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device)
print('Model frozen:', MODEL_ID, MODEL_REV, '| Device:',device)
print('Native label mapping:',model.config.id2label)
print('Processor:',processor.to_json_string()[:1200])
def normalize(s):
    s=str(s).lower().strip().replace(' ','_')
    return {'anger':'angry','happiness':'happy','sadness':'sad','surprised':'surprise',
            'fearful':'fear','neutrality':'neutral','disgusted':'disgust'}.get(s,s)
def predict(path):
    with Image.open(path) as im:
        rgb=im.convert('RGB')
        batch=processor(images=rgb,return_tensors='pt')
    with torch.inference_mode():
        outputs=model(**{k:v.to(device) for k,v in batch.items()})
        index=int(outputs.logits.argmax(dim=-1).item())
    return normalize(model.config.id2label[index])
rows=[]
for sid,expected in originals.itertuples(index=False):
    actual=predict(N07/'sources'/f'{sid}.png')
    rows.append({'source_id':sid,'expected':normalize(expected),'observed':actual,'matches':actual==normalize(expected)})
check=pd.DataFrame(rows)
check.to_csv(OUT/'private_original_prediction_replay.csv',index=False)
print(check[['source_id','expected','observed','matches']].to_string(index=False))
print('ORIGINAL REPLAY MATCH:',int(check.matches.sum()),'/12')


In [ ]:
# Fail CLOSED on any mismatch. A different preprocessing checkpoint cannot be silently used.
ready=bool(check.matches.all())
gate={'source_count':12,'original_prediction_matches':int(check.matches.sum()),
      'original_prediction_total':12,'candidate_model':MODEL_ID,
      'candidate_revision':MODEL_REV,'model_code_provenance':'PENDING_N07_SOURCE_INSPECTION',
      'status':'PROVISIONAL_PREDICTION_REPLAY_PASS' if ready else 'BLOCKED_ORIGINAL_REPLAY_MISMATCH'}
(OUT/'fer_replay_gate.json').write_text(json.dumps(gate,indent=2))
assert ready, 'STOP: this is not a validated replay of N07 original FER predictions. Inspect N07 classifier implementation/preprocessing before doing N17 scoring.'
print('Prediction replay gate passed. Proceed with N17 evaluation; code-level provenance still needs audit.')


In [ ]:
output_csv=OUT/'n17_fer_predictions_PRIVATE.csv'
cached=pd.read_csv(output_csv) if output_csv.exists() else pd.DataFrame()
if len(cached):
    assert not cached.duplicated(['source_id','seed']).any()
generated=cached.to_dict('records') if len(cached) else []
for index,r in enumerate(receipts.sort_values(['source_id','seed']).itertuples(index=False),1):
    existing=[x for x in generated if x['source_id']==r.source_id and int(x['seed'])==r.seed]
    if existing:
        assert len(existing)==1 and existing[0]['output_sha256']==r.output_sha256
        continue
    pred=predict(r.output_path)
    generated.append({'source_id':r.source_id,'seed':int(r.seed),
                      'output_sha256':r.output_sha256,
                      'n17_emotion_predicted_class':pred,
                      'fer_candidate_model':MODEL_ID,'fer_candidate_revision':MODEL_REV,
                      'fer_evaluator_provenance':'PROVISIONAL_MATCHED_ORIGINALS_NOT_YET_N07_CODE_AUDITED'})
    pd.DataFrame(generated).to_csv(output_csv,index=False)
    print(f'FER {index}/24 saved',flush=True)
new=pd.read_csv(output_csv)
assert len(new)==24 and not new.duplicated(['source_id','seed']).any()
paired=ref.merge(new,on=['source_id','seed','output_sha256'],validate='one_to_one')
paired['n17_emotion_preserved']=paired.n17_emotion_predicted_class.map(normalize).eq(paired.original_fer_predicted_class.map(normalize))
paired['delta_emotion']=paired.n17_emotion_preserved.astype(int)-paired.n07_emotion_preserved.astype(int)
grouped=[g.delta_emotion.to_numpy(dtype=float) for _,g in paired.groupby('source_id',sort=True)]
rng=np.random.default_rng(20020)
vals=[]
for i in range(10000):
    picks=rng.integers(0,len(grouped),len(grouped))
    vals.append(np.concatenate([grouped[j] for j in picks]).mean())
summary={'n_sources':12,'n_pairs':24,'n07_fer_preservation':float(paired.n07_emotion_preserved.mean()),
         'n17_fer_preservation':float(paired.n17_emotion_preserved.mean()),
         'paired_delta':float(paired.delta_emotion.mean()),
         'ci95':list(map(float,np.quantile(vals,[.025,.975]))),
         'evaluator_verification':'PROVISIONAL_ONLY_PENDING_EXACT_N07_SOURCE_CODE_AUDIT'}
paired.to_csv(OUT/'private_emotion_paired_rows.csv',index=False)
(OUT/'emotion_provisional_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('STOP: do not label this FINAL or VERIFIED until N07 exact preprocessing and classifier provenance have been audited.')
